<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/PINN_DeepXDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install deepxde

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation $p_t = \kappa p_{xx}$ for $x \in [0, \pi]$, $t \in [0, 1]$, with initial condition $p(x, 0) = \sin(x)$ and boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [16]:
import deepxde as dde
import numpy as np
from deepxde.backend import tf

geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

kappa = 1.0
def pde(x, p):
    p_t = dde.grad.jacobian(p, x, i=0, j=1)
    p_xx = dde.grad.hessian(p, x, i=0, j=0)
    return p_t - kappa * p_xx

net = dde.nn.FNN([2] + [20] * 2 + [1], "tanh", "Glorot uniform")

def output_transform(x, y):
    x_col = x[:, 0:1]
    t_col = x[:, 1:2]
    return tf.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

net.apply_output_transform(output_transform)

data = dde.data.TimePDE(
    geomtime, pde, [],
    num_domain=100,
    num_test=None
)

model = dde.Model(data, net)
model.compile("adam", lr=1e-3)

resampler = dde.callbacks.PDEPointResampler(period=1)
losshistory, train_state = model.train(iterations=1000, callbacks=[resampler], display_every=50)

Compiling model...
'compile' took 0.004039 s

Training model...

Step      Train loss    Test loss     Test metric
0         [3.36e-01]    [3.36e-01]    []  
50        [2.65e-02]    [2.65e-02]    []  
100       [4.20e-03]    [4.20e-03]    []  
150       [3.13e-03]    [3.13e-03]    []  
200       [2.56e-03]    [2.56e-03]    []  
250       [2.10e-03]    [2.10e-03]    []  
300       [1.73e-03]    [1.73e-03]    []  
350       [1.43e-03]    [1.43e-03]    []  
400       [1.19e-03]    [1.19e-03]    []  
450       [9.91e-04]    [9.91e-04]    []  
500       [8.28e-04]    [8.28e-04]    []  
550       [6.95e-04]    [6.95e-04]    []  
600       [5.85e-04]    [5.85e-04]    []  
650       [4.94e-04]    [4.94e-04]    []  
700       [4.18e-04]    [4.18e-04]    []  
750       [3.57e-04]    [3.57e-04]    []  
800       [3.06e-04]    [3.06e-04]    []  
850       [2.64e-04]    [2.64e-04]    []  
900       [2.29e-04]    [2.29e-04]    []  
950       [2.01e-04]    [2.01e-04]    []  
1000      [1.77e-04]    [

In [17]:
import plotly.express as px
import pandas as pd
import numpy as np

domain_x = [0, np.pi]
domain_t = [0, 1]

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
    x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
    t_plot = t_val * np.ones_like(x_plot)
    inputs_plot = np.concatenate([x_plot, t_plot], axis=1)

    p_plot = model.predict(inputs_plot)

    p_exact = np.exp(-t_val) * np.sin(x_plot)
    df_temp = pd.DataFrame({
        'x': x_plot.flatten(),
        'PINN': p_plot.flatten(),
        'Exact': p_exact.flatten(),
        'Time': t_val
    })
    plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')

fig = px.line(df_melted, x='x', y='p', animation_frame='Time',
              color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(xaxis_title='x', yaxis_title='p(x, t)')
fig.show()

## Loss:

In [18]:
import plotly.graph_objects as go
import numpy as np

losses = np.array(losshistory.loss_train).flatten()
steps = np.array(losshistory.steps)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=losses, mode='lines+markers', name='Dirichlet'))
fig.update_layout(
    title='Сходимость обучения PINN (DeepXDE)',
    xaxis_title='Итерация',
    yaxis_title='Функция потерь',
    yaxis_type='log',
    template='plotly_white'
)
fig.show()